In [ ]:
pip install mcap-ros2-support opencv-python rosbags

In [9]:
import os
import cv2
import csv
import numpy as np
from rosbags.rosbag2 import Reader
from rosbags.typesys import stores
from rosbags.typesys.stores import Stores

# Absolute base path pointing to your Windows Desktop
desktop_path = r"C:\Users\Narender\Desktop"

# Defining the 3 distinct output folders directly on the desktop
color_output_dir = os.path.join(desktop_path, "extracted_images")
depth_output_dir = os.path.join(desktop_path, "extracted_depth")
lidar_output_dir = os.path.join(desktop_path, "extracted_lidar")

# Create the 3 individual directories automatically
os.makedirs(color_output_dir, exist_ok=True)
os.makedirs(depth_output_dir, exist_ok=True)
os.makedirs(lidar_output_dir, exist_ok=True)

# Define the absolute path for the final LiDAR CSV file inside its own folder
lidar_output_csv = os.path.join(lidar_output_dir, "lidar_data.csv")

print("Targeting Desktop Rosbag data...")
print("Starting comprehensive data extraction into 3 separate folders...")

# Fetch the exact ROS2 Foxy message profiles used by the LIMO PRO
typestore = stores.get_typestore(Stores.ROS2_FOXY)

color_count = 0
depth_count = 0
lidar_count = 0

# Open the dedicated CSV file inside the LiDAR directory
with open(lidar_output_csv, mode='w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(["Timestamp", "Angle_Min", "Angle_Max", "Angle_Increment", "Ranges_Array"])
    
    # Initialize the binary reader directly over the desktop workspace
    with Reader(desktop_path) as reader:
        for connection, timestamp, rawdata in reader.messages():
            
            # --- 1. EXTRACT COLOR IMAGES ---
            if connection.topic == '/camera/color/image_raw':
                msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
                
                # Reconstruct image matrix from raw byte array
                img = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, 3)
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
                
                # Save frame directly to the color folder
                cv2.imwrite(os.path.join(color_output_dir, f"frame_{color_count:04d}.png"), img)
                color_count += 1
            
            # --- 2. EXTRACT DEPTH CAMERA MAPS ---
            elif connection.topic == '/camera/depth/image_raw':
                msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
                
                # Unpack the 16-bit unsigned depth data (millimeter distances)
                depth_img = np.frombuffer(msg.data, dtype=np.uint16).reshape(msg.height, msg.width)
                
                # Normalize 16-bit data to an 8-bit array (0-255) so human eyes can view it
                visible_depth = cv2.normalize(depth_img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
                
                # Apply a high-contrast colormap (thermal style) for visual clarity
                color_mapped_depth = cv2.applyColorMap(visible_depth, cv2.COLORMAP_JET)
                
                # Save depth image directly to the depth folder
                cv2.imwrite(os.path.join(depth_output_dir, f"depth_{depth_count:04d}.png"), color_mapped_depth)
                depth_count += 1
            
            # --- 3. EXTRACT LIDAR LASER SCANS ---
            elif connection.topic == '/scan':
                msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
                
                # Compress the float array of distance ranges into a clean string
                ranges_str = ",".join([f"{r:.3f}" for r in msg.ranges])
                
                # Write row matching chronological timestamp
                csv_writer.writerow([timestamp, msg.angle_min, msg.angle_max, msg.angle_increment, ranges_str])
                lidar_count += 1

print("\n--- ALL DATA UNPACKED & SORTED SUCCESSFULLY ---")
print(f"Folder 1 [Color Camera]: Unpacked {color_count} images to: {color_output_dir}")
print(f"Folder 2 [Depth Camera]: Unpacked {depth_count} depth maps to: {depth_output_dir}")
print(f"Folder 3 [LiDAR Sensor]: Logged {lidar_count} scans to: {lidar_output_csv}")

Targeting Desktop Rosbag data...
Starting comprehensive data extraction into 3 separate folders...

--- ALL DATA UNPACKED & SORTED SUCCESSFULLY ---
Folder 1 [Color Camera]: Unpacked 1346 images to: C:\Users\Narender\Desktop\extracted_images
Folder 2 [Depth Camera]: Unpacked 1350 depth maps to: C:\Users\Narender\Desktop\extracted_depth
Folder 3 [LiDAR Sensor]: Logged 451 scans to: C:\Users\Narender\Desktop\extracted_lidar\lidar_data.csv
